# **RQ1 :** Tracking Quality Reliability Evaluation

1. Tracking Coverage Left and Right with raw tracking data vs processed Data
2. report the median gap length and % of interpolated frames


In [1]:
#imports
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
# paths
raw_data_path = "../data/raw/output_dataframes/"
processed_data_path = "../data/processed/landmark_dataframes/"

raw_files = [f for f in os.listdir(raw_data_path) if '30fps' in f and '2024' in f and f.endswith('.pkl')]
processed_files = [f for f in os.listdir(processed_data_path) if '30fps' in f and '2024' in f and f.endswith('.pkl')]

# remove one surgon's data
raw_files = [f for f in raw_files if '2024-01-17_17-09-36' not in f and '2024-01-17_18-24-28' not in f and '2024-01-17_18-43-42' not in f]
processed_files = [f for f in processed_files if '2024-01-17_17-09-36' not in f and '2024-01-17_18-24-28' not in f and '2024-01-17_18-43-42' not in f]

print(f"Found {len(raw_files)} raw files and {len(processed_files)} processed files.")

Found 83 raw files and 83 processed files.


In [26]:
raw_coverages_left = []
raw_coverages_right = []

for filename in tqdm(raw_files):
    # Load raw tracking data
    df_raw = pd.read_pickle(os.path.join(raw_data_path, filename))

    # swaped label on purpose (mediapipe gets it wrong)
    df_raw_right = df_raw[df_raw['hand_label'] == 'Left']
    df_raw_left = df_raw[df_raw['hand_label'] == 'Right']

    right_coverage = df_raw_right['frame'].nunique() / (df_raw['frame'].max() - df_raw['frame'].min() + 1)
    left_coverage = df_raw_left['frame'].nunique() / (df_raw['frame'].max() - df_raw['frame'].min() + 1)

    raw_coverages_right.append(right_coverage)
    raw_coverages_left.append(left_coverage)

100%|██████████| 83/83 [00:45<00:00,  1.82it/s]


In [27]:
processed_coverages_left = []
processed_coverages_right = []

fracs_continous_traj_left = []
fracs_continous_traj_right = []

perc_interpolated_left_list = []
perc_interpolated_right_list = []

for filename in tqdm(processed_files):
    # Load processed tracking data
    df_processed = pd.read_pickle(os.path.join(processed_data_path, filename))

    df_processed_right = df_processed[df_processed['hand_label'] == 'Right']
    df_processed_left = df_processed[df_processed['hand_label'] == 'Left']

    right_coverage = df_processed_right['frame'].nunique() / (df_processed['frame'].max() - df_processed['frame'].min() + 1)
    left_coverage = df_processed_left['frame'].nunique() / (df_processed['frame'].max() - df_processed['frame'].min() + 1)

    processed_coverages_right.append(right_coverage)
    processed_coverages_left.append(left_coverage)

    # fraction of tracked frames in segments longer than 1.5sec
    seg_lengths_right = df_processed_right.groupby('segment_id').size()
    seg_lengths_left = df_processed_left.groupby('segment_id').size()

    # at least 1.5 sec (45 frames at 30fps)
    frac_right = seg_lengths_right[seg_lengths_right >= 45].sum() / df_processed_right['frame'].nunique()
    frac_left = seg_lengths_left[seg_lengths_left >= 45].sum() / df_processed_left['frame'].nunique()
    fracs_continous_traj_right.append(frac_right)
    fracs_continous_traj_left.append(frac_left)

    # percentage of interpolated frames
    perc_interpolated_right = df_processed_right['hand_score'].isna().sum() / len(df_processed_right)
    perc_interpolated_left = df_processed_left['hand_score'].isna().sum() / len(df_processed_left)
    perc_interpolated_right_list.extend([perc_interpolated_right])
    perc_interpolated_left_list.extend([perc_interpolated_left])

100%|██████████| 83/83 [00:05<00:00, 14.19it/s]


In [28]:
# raw vs processed coverage
print(f"Raw left hand coverage: {np.mean(raw_coverages_left):.2%} ± {np.std(raw_coverages_left):.2%}")
print(f"Raw right hand coverage: {np.mean(raw_coverages_right):.2%} ± {np.std(raw_coverages_right):.2%}")

print(f"Processed left hand coverage: {np.mean(processed_coverages_left):.2%} ± {np.std(processed_coverages_left):.2%}")
print(f"Processed right hand coverage: {np.mean(processed_coverages_right):.2%} ± {np.std(processed_coverages_right):.2%}")

Raw left hand coverage: 81.83% ± 9.44%
Raw right hand coverage: 78.82% ± 6.43%
Processed left hand coverage: 82.71% ± 9.25%
Processed right hand coverage: 79.11% ± 6.41%


In [9]:
# number of interpolated frames
print(f"Percentage of interpolated frames (left hand): {np.mean(perc_interpolated_left_list):.4f} ± {np.std(perc_interpolated_left_list):.4f}")
print(f"Percentage of interpolated frames (right hand): {np.mean(perc_interpolated_right_list):.4f} ± {np.std(perc_interpolated_right_list):.4f}")

Percentage of interpolated frames (left hand): 0.0132 ± 0.0090
Percentage of interpolated frames (right hand): 0.0067 ± 0.0041


In [ ]:
# median gap length distribution
print("Fraction of continous trajcectories (frames belonging to segments longer than 1.5 sec / total_tracked frames):")
print(f"Left hand):\n {np.mean(fracs_continous_traj_left):.2%} ± {np.std(fracs_continous_traj_left):.2%}")
print(f"Right hand):\n {np.mean(fracs_continous_traj_right):.2%} ± {np.std(fracs_continous_traj_right):.2%}")

Distribution of continous trajectory fraction (left hand):
 count    83.000000
mean      0.960031
std       0.026706
min       0.878300
25%       0.943754
50%       0.966001
75%       0.979692
max       0.997812
dtype: float64
Distribution of continous trajectory fraction (right hand):
 count    83.000000
mean      0.970491
std       0.016770
min       0.893306
25%       0.964307
50%       0.973709
75%       0.981999
max       0.996821
dtype: float64
